# Phase 1: Python EDA + Business KPI Validation
# Load data

In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

claims = pd.read_csv(r"C:\Business-Data-Projects\StarHealth\fraud_detection\data\claims_full_dataset.csv")
customers = pd.read_csv(r"C:\Business-Data-Projects\StarHealth\fraud_detection\data\customers_full_dataset.csv")
hospitals = pd.read_csv(r"C:\Business-Data-Projects\StarHealth\fraud_detection\data\hospitals_full_dataset.csv")

df = claims.merge(customers, on="customer_id", how="left")
df = df.merge(hospitals, on="hospital_id", how="left")

df.head()

,claim_id,customer_id,policy_id,hospital_id,claim_amount,claim_status,disease_category,days_hospitalized,fraud_flag,claim_date,...,paid_timestamp,premium_amount,age,gender,city,income_band,hospital_name,hospital_city,network_hospital,onboarding_date
0,CLM00001,CUST0271,POL02612,HOSP036,33216.79,Rejected,Respiratory,1,0,2025-07-05,...,2025-07-12 09:00:00,58278,41,Female,Delhi,Low,Hospital 36,Pune,1,2024-11-18
1,CLM00002,CUST0043,POL02462,HOSP025,83280.31,Rejected,Respiratory,6,0,2025-09-04,...,2025-09-11 10:00:00,9780,69,Female,Bengaluru,Middle,Hospital 25,Pune,1,2023-10-31
2,CLM00003,CUST0432,POL00880,HOSP037,75150.89,Approved,Cancer,7,0,2024-04-20,...,2024-04-29 01:00:00,34107,42,Female,Chennai,Middle,Hospital 37,Mumbai,1,2025-02-19
3,CLM00004,CUST0343,POL00993,HOSP001,23820.08,Under Review,Cardiac,3,0,2025-01-19,...,2025-01-26 04:00:00,39587,66,Female,Hyderabad,Middle,Hospital 1,Lucknow,1,2025-03-12
4,CLM00005,CUST0326,POL00001,HOSP015,94668.96,Paid,Orthopedic,12,1,2024-07-09,...,2024-07-17 08:00:00,50192,49,Female,Lucknow,High,Hospital 15,Jaipur,1,2024-08-10


# Create KPI fields

In [53]:
df["claim_per_day"] = df["claim_amount"] / df["days_hospitalized"]

df["high_claim_flag"] = np.where(
    df["claim_amount"] > df["claim_amount"].quantile(0.75),
    1, 0
)

df["fraud_loss_amount"] = np.where(
    df["fraud_flag"] == 1,
    df["claim_amount"],
    0
)
display(df[["claim_amount", "claim_per_day", "high_claim_flag", "fraud_loss_amount"]].head())

,claim_amount,claim_per_day,high_claim_flag,fraud_loss_amount
0,33216.79,33216.790000,0,0.00
1,83280.31,13880.051667,1,0.00
2,75150.89,10735.841429,1,0.00
3,23820.08,7940.026667,0,0.00
4,94668.96,7889.080000,1,94668.96


# KPI Summary

In [55]:
total_claims = len(df)
fraud_claims = df["fraud_flag"].sum()
fraud_rate = fraud_claims / total_claims
fraud_loss = df["fraud_loss_amount"].sum()
avg_fraud_claim = df[df["fraud_flag"] == 1]["claim_amount"].mean()
successful_settlement_rate = len(df[df["claim_status"] == "Approved"]) / total_claims

print("Total Claims:", total_claims)
print("Fraud Claims:", fraud_claims)
print("Fraud Rate:", fraud_rate)
print("Fraud Loss Amount:", fraud_loss)
print("Avg Fraud Claim:", avg_fraud_claim)
print("Settlement Rate:", successful_settlement_rate)

Total Claims: 2120
Fraud Claims: 223
Fraud Rate: 0.10518867924528302
Fraud Loss Amount: 16078534.34
Avg Fraud Claim: 72101.05085201794
Settlement Rate: 0.25471698113207547


# Suspicious Hospitals

In [11]:
hospital_risk = df.groupby(["hospital_id", "hospital_name"]).agg(
    total_claims=("claim_id", "count"),
    fraud_claims=("fraud_flag", "sum"),
    avg_claim_amount=("claim_amount", "mean"),
    avg_claim_per_day=("claim_per_day", "mean"),
    fraud_loss=("fraud_loss_amount", "sum")
).reset_index()

hospital_risk["fraud_rate"] = 100* hospital_risk["fraud_claims"] / hospital_risk["total_claims"]

hospital_risk.sort_values(
    ["fraud_rate", "fraud_loss"],
    ascending=False
).head(10)

,hospital_id,hospital_name,total_claims,fraud_claims,avg_claim_amount,avg_claim_per_day,fraud_loss,fraud_rate
29,HOSP030,Hospital 30,19,6,48618.436842,15306.908888,433413.75,31.578947
38,HOSP039,Hospital 39,23,7,56835.008261,8740.365480,586941.88,30.434783
94,HOSP095,Hospital 95,25,7,49712.326800,11038.588986,465494.31,28.000000
34,HOSP035,Hospital 35,16,4,48766.843750,14272.917923,277538.42,25.000000
69,HOSP070,Hospital 70,12,3,42689.151667,19164.110190,168902.83,25.000000
77,HOSP078,Hospital 78,21,5,49971.687619,12427.339405,427795.68,23.809524
32,HOSP033,Hospital 33,17,4,46967.856471,9903.731884,326434.67,23.529412
90,HOSP091,Hospital 91,17,4,52845.422941,10813.829438,250860.04,23.529412
81,HOSP082,Hospital 82,22,5,54061.280909,17515.868089,462727.78,22.727273
43,HOSP044,Hospital 44,18,4,47124.301111,11437.615192,174648.20,22.222222


# Business Insights
Hospital 30 has high fraud rate (31.57%) and high claim-per-day (15306.908), suggesting possible inflated billing.

# Phase 2:  Advance Analysis
Funnel + cohort + retention analysis

A. Claims Funnel Analysis

In [16]:
stage_order = [
    "Submitted",
    "Document Verified",
    "Approved",
    "Paid"
]

df["stage"] = np.select(
    [
        df["claim_status"] == "Submitted",
        df["claim_status"] == "Under Review",   # maps to Document Verified
        df["claim_status"] == "Approved",
        df["claim_status"] == "Paid"
    ],
    stage_order,
    default="Rejected"   # all rejected claims
)

# Check mapping
display(df[["claim_status", "stage"]].drop_duplicates())

,claim_status,stage
0,Rejected,Rejected
2,Approved,Approved
3,Under Review,Document Verified
4,Paid,Paid


# Fraud leakage through approved claims

In [30]:
fraud_leakage = df[
    (df["fraud_flag"] == 1) &
    (df["claim_status"] == "Approved")
]

leakage_rate = 100* len(fraud_leakage) / max(df["fraud_flag"].sum(), 1)

print("Fraud Leakage Rate:", leakage_rate)

Fraud Leakage Rate: 46.63677130044843


46.636 % of suspicious claims were approved, indicating process leakage.

B. Cohort Analysis

claim_date
onboarding_date

In [20]:
df["claim_date"] = pd.to_datetime(df["claim_date"])
df["onboarding_date"] = pd.to_datetime(df["onboarding_date"])

df["claim_month"] = df["claim_date"].dt.to_period("M").astype(str)
df["hospital_onboarding_month"] = df["onboarding_date"].dt.to_period("M").astype(str)

# Fraud by claim month

In [21]:
claim_month_cohort = df.groupby("claim_month").agg(
    total_claims=("claim_id", "count"),
    fraud_claims=("fraud_flag", "sum"),
    fraud_loss=("fraud_loss_amount", "sum")
).reset_index()

claim_month_cohort["fraud_rate"] = (
    claim_month_cohort["fraud_claims"] / claim_month_cohort["total_claims"]
)

claim_month_cohort

,claim_month,total_claims,fraud_claims,fraud_loss,fraud_rate
0,2024-01,87,3,291907.26,0.034483
1,2024-02,63,4,272111.89,0.063492
2,2024-03,93,9,821158.05,0.096774
3,2024-04,91,12,979402.00,0.131868
4,2024-05,101,12,769833.41,0.118812
5,2024-06,96,11,783548.40,0.114583
6,2024-07,98,11,981723.56,0.112245
7,2024-08,81,11,952641.57,0.135802
8,2024-09,95,10,494903.34,0.105263
9,2024-10,91,12,665213.70,0.131868


# Fraud by hospital onboarding month

In [29]:
hospital_cohort = df.groupby("hospital_onboarding_month").agg(
    total_claims=("claim_id", "count"),
    fraud_claims=("fraud_flag", "sum"),
    fraud_loss=("fraud_loss_amount", "sum")
).reset_index()

hospital_cohort["fraud_rate"] = (
    100* hospital_cohort["fraud_claims"] / hospital_cohort["total_claims"]
)
hospital_cohort.sort_values(
    ["fraud_rate", "fraud_claims"],
    ascending=False
).head(10)



,hospital_onboarding_month,total_claims,fraud_claims,fraud_loss,fraud_rate
12,2024-01,16,3,379285.88,18.750000
23,2024-12,61,11,857155.73,18.032787
27,2025-04,99,16,1353634.14,16.161616
3,2023-04,83,13,1129680.51,15.662651
2,2023-03,100,14,848781.61,14.000000
0,2023-01,89,12,837444.33,13.483146
6,2023-07,15,2,91412.77,13.333333
18,2024-07,15,2,142315.50,13.333333
22,2024-11,46,6,363933.00,13.043478
20,2024-09,86,11,632547.98,12.790698


Business Insight : Recently onboarded hospitals are showing materially higher fraud rates than older cohorts.
Interview Explanation :
Cohort analysis by hospital onboarding month showed that recently onboarded hospitals, particularly the April 2025 cohort, had fraud rates above 16% and generated over ₹13.5 lakh in fraud loss. This suggests that fraud risk is highest during the early provider lifecycle, so I would recommend tighter onboarding controls and enhanced monitoring during the first 90 days.

c) Retention Analysis
Do fraud customers repeat within 30/60/90 days?

In [31]:
df["claim_date"] = pd.to_datetime(df["claim_date"])

fraud_df = df[df["fraud_flag"] == 1].sort_values(
    ["customer_id", "claim_date"]
)

fraud_df["previous_fraud_date"] = fraud_df.groupby("customer_id")["claim_date"].shift(1)

fraud_df["days_since_previous_fraud"] = (
    fraud_df["claim_date"] - fraud_df["previous_fraud_date"]
).dt.days

fraud_df["repeat_30d"] = fraud_df["days_since_previous_fraud"].between(1, 30)
fraud_df["repeat_60d"] = fraud_df["days_since_previous_fraud"].between(1, 60)
fraud_df["repeat_90d"] = fraud_df["days_since_previous_fraud"].between(1, 90)

repeat_summary = fraud_df.agg(
    repeat_30d=("repeat_30d", "sum"),
    repeat_60d=("repeat_60d", "sum"),
    repeat_90d=("repeat_90d", "sum")
)

repeat_summary

,repeat_30d,repeat_60d,repeat_90d
repeat_30d,104.0,NaN,NaN
repeat_60d,NaN,106.0,NaN
repeat_90d,NaN,NaN,108.0


Business Insight:
Repeat fraud within 30 days indicates organized fraud behavior or weak customer-level blocking.
Retention analysis showed that 104 out of 108 repeat fraud customers, or over 96%, submitted another fraudulent claim within 30 days of the previous one. This indicates that fraudsters tend to reoffend quickly, so I would recommend automatically routing any subsequent claim within 30 days to manual review.

# Phase 3 : ML Fraud prediction Model : Use ML to predict fraud_flag

A. Select Features

In [39]:
model_df = df.copy()

features = [
    "claim_amount",
    "days_hospitalized",
    "disease_category",
    "network_hospital",
    "hospital_city",
]

target = "fraud_flag"

B. Encoding

In [40]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

X = model_df[features]
y = model_df[target]

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

C. Train test split

In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

D. Random Forest Model

In [77]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ))
])

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.95      0.99      0.97       474
           1       0.88      0.52      0.65        56

    accuracy                           0.94       530
   macro avg       0.91      0.75      0.81       530
weighted avg       0.94      0.94      0.93       530

[[470   4]
 [ 27  29]]
ROC AUC: 0.7838117842073538


E. Create Fraud Risk score

In [78]:
model_df["fraud_probability"] = 100*rf_model.predict_proba(X)[:, 1]

model_df["fraud_risk_segment"] = pd.cut(
    model_df["fraud_probability"],
    bins=[0, 0.4, 0.7, 1],
    labels=["Low Risk", "Medium Risk", "High Risk"]
)

model_df[[
    "claim_id",
    "customer_id",
    "hospital_id",
    "claim_amount",
    "fraud_probability",
    "fraud_risk_segment"
]].head()

,claim_id,customer_id,hospital_id,claim_amount,fraud_probability,fraud_risk_segment
0,CLM00001,CUST0271,HOSP036,33216.79,5.0,NaN
1,CLM00002,CUST0043,HOSP025,83280.31,12.0,NaN
2,CLM00003,CUST0432,HOSP037,75150.89,0.0,NaN
3,CLM00004,CUST0343,HOSP001,23820.08,0.5,Medium Risk
4,CLM00005,CUST0326,HOSP015,94668.96,96.0,NaN


| Risk Segment | Recommended Action         |
| ------------ | -------------------------- |
| Low Risk     | Auto-approve               |
| Medium Risk  | Additional validation      |
| High Risk    | Manual fraud investigation |

Business Insight :
The Random Forest model assigned each claim a fraud probability and categorized claims into Low, Medium, and High Risk segments. High Risk claims, such as CLM00005 with a 96% fraud probability, should be routed to manual investigation before approval. This allows the company to auto-approve low-risk claims while concentrating fraud review efforts on the most suspicious and financially material claims.

F. Save output for Tableau

In [ ]:
model_df.to_csv(r"C:\Business-Data-Projects\StarHealth\fraud_detection\output\fraud_detection_output.csv", index=False)
hospital_risk.to_csv(r"C:\Business-Data-Projects\StarHealth\fraud_detection\output\hospital_risk_summary.csv", index=False)
claim_month_cohort.to_csv(r"C:\Business-Data-Projects\StarHealth\fraud_detection\output\claim_month_cohort.csv", index=False)
display(model_df)
display(hospital_risk)
display(claim_month_cohort)
# total_claims = model_df["claim_id"].count()
# print("Total Claims ", total_claims)
# fraud_claims = model_df["fraud_flag"].sum()
# print("Total Fraud Claims ", fraud_claims)

# Leakage_rate = (
#     model_df[
#         (model_df["fraud_flag"] == 1) &
#         (model_df["claim_status"] == "Approved")
#     ].shape[0]
#     /
#     model_df["fraud_flag"].sum()
# )
# print("Leakage Rate:", Leakage_rate)

,claim_id,customer_id,policy_id,hospital_id,claim_amount,claim_status,disease_category,days_hospitalized,fraud_flag,claim_date,...,hospital_city,network_hospital,onboarding_date,claim_per_day,high_claim_flag,fraud_loss_amount,claim_month,hospital_onboarding_month,fraud_probability,fraud_risk_segment
0,CLM00001,CUST0271,POL02612,HOSP036,33216.79,Rejected,Respiratory,1,0,2025-07-05,...,Pune,1,2024-11-18,33216.790000,0,0.00,2025-07,2024-11,5.0,NaN
1,CLM00002,CUST0043,POL02462,HOSP025,83280.31,Rejected,Respiratory,6,0,2025-09-04,...,Pune,1,2023-10-31,13880.051667,1,0.00,2025-09,2023-10,12.0,NaN
2,CLM00003,CUST0432,POL00880,HOSP037,75150.89,Approved,Cancer,7,0,2024-04-20,...,Mumbai,1,2025-02-19,10735.841429,1,0.00,2024-04,2025-02,0.0,NaN
3,CLM00004,CUST0343,POL00993,HOSP001,23820.08,Under Review,Cardiac,3,0,2025-01-19,...,Lucknow,1,2025-03-12,7940.026667,0,0.00,2025-01,2025-03,0.5,Medium Risk
4,CLM00005,CUST0326,POL00001,HOSP015,94668.96,Paid,Orthopedic,12,1,2024-07-09,...,Jaipur,1,2024-08-10,7889.080000,1,94668.96,2024-07,2024-08,96.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2115,CLM02116,CUST0216,POL00792,HOSP022,12867.10,Paid,Cancer,3,1,2026-01-22,...,Hyderabad,1,2023-10-14,4289.033333,0,12867.10,2026-01,2023-10,70.0,NaN
2116,CLM02117,CUST0216,POL01395,HOSP047,16444.07,Paid,Cardiac,4,1,2026-02-21,...,Pune,0,2024-05-08,4111.017500,0,16444.07,2026-02,2024-05,65.5,NaN
2117,CLM02118,CUST0403,POL01651,HOSP085,55946.11,Approved,Cancer,6,1,2025-05-23,...,Chennai,1,2023-03-26,9324.351667,0,55946.11,2025-05,2023-03,3.0,NaN
2118,CLM02119,CUST0403,POL02486,HOSP029,37736.46,Paid,Cancer,7,1,2025-06-22,...,Lucknow,0,2023-08-08,5390.922857,0,37736.46,2025-06,2023-08,61.0,NaN


,hospital_id,hospital_name,total_claims,fraud_claims,avg_claim_amount,avg_claim_per_day,fraud_loss,fraud_rate
0,HOSP001,Hospital 1,20,3,41159.856500,12970.138612,147609.98,15.000000
1,HOSP002,Hospital 2,15,2,44726.339333,11289.627425,91412.77,13.333333
2,HOSP003,Hospital 3,19,0,40418.957895,8199.193833,0.00,0.000000
3,HOSP004,Hospital 4,22,4,49199.668636,11647.794614,304505.86,18.181818
4,HOSP005,Hospital 5,18,2,42404.866667,11742.013554,63388.66,11.111111
...,...,...,...,...,...,...,...,...
95,HOSP096,Hospital 96,29,5,49218.426207,12375.431464,413361.07,17.241379
96,HOSP097,Hospital 97,17,1,39632.730588,9419.057540,57688.35,5.882353
97,HOSP098,Hospital 98,20,4,53732.649000,20707.311803,334067.87,20.000000
98,HOSP099,Hospital 99,19,0,45220.691579,16125.650207,0.00,0.000000


,claim_month,total_claims,fraud_claims,fraud_loss,fraud_rate
0,2024-01,87,3,291907.26,0.034483
1,2024-02,63,4,272111.89,0.063492
2,2024-03,93,9,821158.05,0.096774
3,2024-04,91,12,979402.00,0.131868
4,2024-05,101,12,769833.41,0.118812
5,2024-06,96,11,783548.40,0.114583
6,2024-07,98,11,981723.56,0.112245
7,2024-08,81,11,952641.57,0.135802
8,2024-09,95,10,494903.34,0.105263
9,2024-10,91,12,665213.70,0.131868


Total Claims  2120
Total Fraud Claims  223
Leakage Rate: 0.4663677130044843


Relationship :
fraud_detection_output -->  hospital_risk_summary  (by hospital_id)

If claim_month_cohort.csv contains one row per claim_month
fraud_detection_output -->  claim_month_cohort     (by claim_month)  
